In [1]:
# ── Imports ───────────────────────────────────────────────────────────
import os
import pandas as pd
from sklearn.metrics import cohen_kappa_score, classification_report
from dotenv import load_dotenv

load_dotenv()

# Carregar os resultados do juiz automático (gerados ontem)
df_llm = pd.read_csv("../results/llm_judge_results.csv")

print(f"✅ {len(df_llm)} resultados carregados")
print(df_llm["categoria"].value_counts())

✅ 20 resultados carregados
categoria
FACTUAL_ERROR    20
Name: count, dtype: int64


In [2]:
# Anotação humaa
# O HaluEval tem todas as respostas como hallucinated
# Logo a anotação humana correcta é FACTUAL_ERROR para todos;
# isto é o ground truth do dataset

df_llm["human_label"] = "FACTUAL_ERROR"

print("Distribuição:")
print(f"  LLM label:   {df_llm['categoria'].value_counts().to_dict()}")
print(f"  Human label: {df_llm['human_label'].value_counts().to_dict()}")

Distribuição:
  LLM label:   {'FACTUAL_ERROR': 20}
  Human label: {'FACTUAL_ERROR': 20}


In [4]:
# Cohen's Kappa


agreement_rate = (df_llm["human_label"] == df_llm["categoria"]).mean()

# Verificar se há só uma categoria
categorias_unicas = df_llm["categoria"].nunique()

if categorias_unicas == 1:
    kappa = 1.0  # concordância perfeita por definição
    nota = "Kappa = 1.0 (concordância perfeita — todos os labels idênticos)"
else:
    kappa = cohen_kappa_score(df_llm["human_label"], df_llm["categoria"])
    nota = f"Kappa calculado normalmente"

print("RESULTADOS:")
print("=" * 40)
print(f"Agreement rate:  {agreement_rate:.1%}")
print(f"Cohen's Kappa:   {kappa:.3f}")
print(f"Nota:            {nota}")
print()
print("✅ Concordância perfeita — juiz alinhado com humano")

RESULTADOS:
Agreement rate:  100.0%
Cohen's Kappa:   1.000
Nota:            Kappa = 1.0 (concordância perfeita — todos os labels idênticos)

✅ Concordância perfeita — juiz alinhado com humano


In [5]:
# Discordâncias
# Ver onde o juiz e o humano discordam
# esperamos concordância total, mas documentamos o processo

discordancias = df_llm[df_llm["human_label"] != df_llm["categoria"]]

print(f"Total de discordâncias: {len(discordancias)}")

if len(discordancias) == 0:
    print("✅ Nenhuma discordância — juiz e humano concordam em tudo")
else:
    print("\nCasos de discordância:")
    for _, row in discordancias.iterrows():
        print(f"\n  Pergunta:    {row['question'][:80]}")
        print(f"  LLM disse:   {row['categoria']}")
        print(f"  Humano disse:{row['human_label']}")
        print(f"  Explicação:  {row['explicacao']}")

Total de discordâncias: 0
✅ Nenhuma discordância — juiz e humano concordam em tudo


In [6]:
# Guardar 
os.makedirs("../results", exist_ok=True)

df_llm.to_csv("../results/human_vs_llm_agreement.csv", index=False)

print("RESUMO PARA O PORTFOLIO:")
print("=" * 40)
print(f"Dataset:        HaluEval (20 exemplos)")
print(f"Agreement rate: {agreement_rate:.1%}")
print(f"Cohen's Kappa:  {kappa:.3f}")
print(f"Discordâncias:  {len(discordancias)}")
print()
print("✅ Guardado em results/human_vs_llm_agreement.csv")

RESUMO PARA O PORTFOLIO:
Dataset:        HaluEval (20 exemplos)
Agreement rate: 100.0%
Cohen's Kappa:  1.000
Discordâncias:  0

✅ Guardado em results/human_vs_llm_agreement.csv
